In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from mlfinpy.util.volatility import get_daily_vol
from mlfinpy.filters.filters import cusum_filter
from mlfinpy.labeling.labeling import get_events, get_bins
from mlfinpy.util.frac_diff import frac_diff_ffd
from mlfinpy.sample_weights.attribution import get_weights_by_return

In [2]:
# 最新のExperimentフォルダを自動特定
BASE_EXP_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "Experiment"))
folders = glob.glob(os.path.join(BASE_EXP_DIR, "Experiment_*"))
max_num = max([int(re.search(r"Experiment_(\d{4})$", os.path.basename(f)).group(1)) for f in folders if os.path.isdir(f)] + [1])
seq_str = f"{max_num:04d}"

SUB_EXP_DIR = os.path.join(BASE_EXP_DIR, f"Experiment_{seq_str}")
train_csv_path = os.path.join(SUB_EXP_DIR, f"train_data_{seq_str}.csv")

print(f"📂 FinRLデータをロード中: {train_csv_path}")
df_finrl = pd.read_csv(train_csv_path)
if 'Unnamed: 0' in df_finrl.columns:
    df_finrl = df_finrl.drop(columns=['Unnamed: 0'])
elif df_finrl.columns[0] == "":
    df_finrl = df_finrl.rename(columns={"": "index_id"})

📂 FinRLデータをロード中: /home/hamano/FinRL/Experiment/Experiment_0021/train_data_0021.csv


In [3]:
print("🔄 銘柄(tic)ごとに mlfinpy 前処理をループ適用中...")
processed_tics = []

for tic, group in df_finrl.groupby("tic"):
    # 時系列順にソートして日付を一時的にインデックス化
    group_sorted = group.sort_values("date").copy()
    group_sorted.index = pd.to_datetime(group_sorted["date"])
    close_series = group_sorted["close"]
    
    # 1. 日次ボラティリティの計算
    daily_vol = get_daily_vol(close_series, lookback=50)
    
    # 2. CUSUMフィルタによるイベント検知
    t_events = cusum_filter(close_series, threshold=max(daily_vol.mean(), 1e-6))
    
    # 3. トリプルバリア法と正解ラベルの抽出
    if len(t_events) > 0:
        events = get_events(close_series, t_events, pt_sl=[1, 1], target=daily_vol, min_ret=0.0, num_threads=1)
        # 4. サンプルの重み付け計算
        events_renamed = events.rename(columns={'t': 't1'})
        events_cleaned = events_renamed.dropna(subset=['t1'])
        events_cleaned = events_cleaned[events_cleaned.index.notnull()]
        
        if not events_cleaned.empty:
            weights = get_weights_by_return(events_cleaned, close_series, num_threads=1)
            group_sorted["mlfin_weight"] = weights.reindex(group_sorted.index).fillna(method='ffill').fillna(1.0)
        else:
            group_sorted["mlfin_weight"] = 1.0
    else:
        group_sorted["mlfin_weight"] = 1.0
        
    # 5. 分数階微分 (FFD)
    ffd_df = frac_diff_ffd(close_series.to_frame(), diff_amt=0.5, thresh=1e-5)
    group_sorted["mlfin_ffd"] = ffd_df.reindex(group_sorted.index).iloc[:, 0].fillna(method='bfill').fillna(0.0)
    
    # インデックスを通常に戻して保持
    group_sorted = group_sorted.reset_index(drop=True)
    processed_tics.append(group_sorted)

# データを統合し、FinRLの並び順に再整列
df_bridge = pd.concat(processed_tics, ignore_index=True)
df_bridge = df_bridge.sort_values(by=["date", "tic"]).reset_index(drop=True)

🔄 銘柄(tic)ごとに mlfinpy 前処理をループ適用中...


/tmp/ipykernel_180570/2390484514.py:26: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  group_sorted["mlfin_weight"] = weights.reindex(group_sorted.index).fillna(method='ffill').fillna(1.0)
/tmp/ipykernel_180570/2390484514.py:34: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  group_sorted["mlfin_ffd"] = ffd_df.reindex(group_sorted.index).iloc[:, 0].fillna(method='bfill').fillna(0.0)
/tmp/ipykernel_180570/2390484514.py:26: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  group_sorted["mlfin_weight"] = weights.reindex(group_sorted.index).fillna(method='ffill').fillna(1.0)
/tmp/ipykernel_180570/2390484514.py:34: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill(

In [5]:
print(f"💾 前処理データをCSVに上書き保存します: {train_csv_path}")
df_bridge.to_csv(train_csv_path, index=True)

# READMEに追記
readme_md_path = os.path.join(SUB_EXP_DIR, f"README_{seq_str}.md")
append_text = "\n### 🧠 MLFinlab 高度前処理 (Jupyter)- **追加インジケータ**: `mlfin_ffd` (分数階微分次数0.5), `mlfin_weight` (サンプル固有重み)\n"
if os.path.exists(readme_md_path):
    with open(readme_md_path, "a", encoding="utf-8") as f:
        f.write(append_text)
print("✨ 橋渡し用データの作成が完全に成功しました！")

💾 前処理データをCSVに上書き保存します: /home/hamano/FinRL/Experiment/Experiment_0021/train_data_0021.csv
✨ 橋渡し用データの作成が完全に成功しました！
